# Image Classification with Convolutional Neural Networks

## CIFAR-10 Dataset Classification using PyTorch

This notebook demonstrates how to build, train, and evaluate a Convolutional Neural Network (CNN) for image classification using the CIFAR-10 dataset.

### Prerequisites

Install the required packages before running the notebook:

```bash
pip install numpy pillow torch torchvision
```

### Notebook Outline

1. **Setup & Imports** - Import libraries and configure data transformations
2. **Data Loading** - Download and prepare CIFAR-10 dataset
3. **Model Architecture** - Define the CNN structure
4. **Training** - Train the model on the training set
5. **Evaluation** - Test the model on the test set
6. **Inference** - Make predictions on custom images

---

**Note:** This is a educational implementation. For production use, consider using pre-trained models via transfer learning (ResNet, VGG, etc.) for better accuracy.

---

## 1. Setup & Imports

Import necessary libraries for deep learning and image processing.

In [ ]:
# Core numerical computing
import numpy as np

# Image processing
from PIL import Image

# PyTorch deep learning framework
import torch
import torch.nn as nn           # Neural network layers
import torch.nn.functional as F # Activation functions and utilities
import torch.optim as optim     # Optimization algorithms

# Torchvision for computer vision utilities
import torchvision
import torchvision.transforms as transforms

In [ ]:
# Data transformation pipeline
# - ToTensor: Converts PIL Image to PyTorch Tensor (scales values to [0, 1])
# - Normalize: Normalizes each channel to [-1, 1] using mean=0.5, std=0.5
#   This helps with faster convergence during training
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

---

## 2. Data Loading

Download and prepare the CIFAR-10 dataset. CIFAR-10 contains 60,000 images across 10 classes: airplanes, cars, birds, cats, deer, dogs, frogs, horses, ships, and trucks.

In [ ]:
# CIFAR-10 dataset configuration
# The dataset will be automatically downloaded if not present in './data'
data_dir = './data'

# Training dataset (50,000 images)
train_data = torchvision.datasets.CIFAR10(
    root=data_dir, 
    train=True, 
    transform=transform, 
    download=True
)

# Test dataset (10,000 images)
test_data = torchvision.datasets.CIFAR10(
    root=data_dir, 
    train=False, 
    transform=transform, 
    download=True
)

# DataLoaders handle batching and shuffling
# batch_size=32: Process 32 images at a time
# shuffle=True: Randomize order each epoch for training
# num_workers=2: Use 2 subprocesses for data loading
train_loader = torch.utils.data.DataLoader(
    train_data, 
    batch_size=32, 
    shuffle=True, 
    num_workers=2
)

test_loader = torch.utils.data.DataLoader(
    test_data, 
    batch_size=32, 
    shuffle=False, 
    num_workers=2
)

print(f"Training samples: {len(train_data)}")
print(f"Test samples: {len(test_data)}")

In [ ]:
# Optional: Debug - Verify dataset files are present
import os

test_path = '/home/ep0/Documents/GitHub/Python_code_ML/ImgClassificationCNN/data/cifar-10-python.tar.gz'
print(f"File exists check: {os.path.exists(test_path)}")
print(f"Current working directory: {os.getcwd()}")
print(f"Listing data dir:")
if os.path.exists('/home/ep0/Documents/GitHub/Python_code_ML/ImgClassificationCNN/data'):
    print(os.listdir('/home/ep0/Documents/GitHub/Python_code_ML/ImgClassificationCNN/data'))
else:
    print("Data directory not found!")

In [37]:
image, label =  train_data[0]

In [38]:
image.size()

torch.Size([3, 32, 32])

In [ ]:
# CIFAR-10 class names (10 categories)
# Used for interpreting model predictions
class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer', 
               'dog', 'frog', 'horse', 'ship', 'truck']

In [ ]:
class NeuralNetwork(nn.Module):
    """
    Convolutional Neural Network for CIFAR-10 image classification.
    
    Architecture:
        Input: (3, 32, 32) - RGB image
        Conv1: 3 -> 12 channels, 5x5 kernel -> (12, 28, 28)
        Pool1: 2x2 max pooling -> (12, 14, 14)
        Conv2: 12 -> 24 channels, 5x5 kernel -> (24, 10, 10)
        Pool2: 2x2 max pooling -> (24, 5, 5)
        Flatten: (24, 5, 5) -> 600
        FC1: 600 -> 120
        FC2: 120 -> 84
        FC3: 84 -> 10 (output classes)
    """
    
    def __init__(self):
        super(NeuralNetwork, self).__init__()
        
        # First convolutional layer
        # Input: 3 channels (RGB), Output: 12 feature maps
        # Kernel size: 5x5, Output spatial size: (32-5+1) = 28
        self.conv1 = nn.Conv2d(3, 12, 5)
        
        # Max pooling layer (reduces spatial dimensions by half)
        # Applied after conv1 and conv2
        self.pool = nn.MaxPool2d(2, 2)
        
        # Second convolutional layer
        # Input: 12 channels, Output: 24 feature maps
        # Kernel size: 5x5, Output spatial size: (14-5+1) = 10
        self.conv2 = nn.Conv2d(12, 24, 5)
        
        # Fully connected layers
        # Input size: 24 * 5 * 5 = 600 (flattened feature maps)
        self.fc1 = nn.Linear(24 * 5 * 5, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)  # 10 output classes

    def forward(self, x):
        """
        Forward pass through the network.
        
        Args:
            x: Input tensor of shape (batch_size, 3, 32, 32)
        
        Returns:
            Output tensor of shape (batch_size, 10)
        """
        # Conv1 -> ReLU -> Pool
        x = self.pool(F.relu(self.conv1(x)))  # (3,32,32) -> (12,14,14)
        
        # Conv2 -> ReLU -> Pool
        x = self.pool(F.relu(self.conv2(x)))  # (12,14,14) -> (24,5,5)
        
        # Flatten all dimensions except batch
        x = torch.flatten(x, 1)  # (batch, 24, 5, 5) -> (batch, 600)
        
        # Fully connected layers with ReLU activation
        x = F.relu(self.fc1(x))  # 600 -> 120
        x = F.relu(self.fc2(x))  # 120 -> 84
        
        # Output layer (no activation, CrossEntropyLoss applies softmax)
        x = self.fc3(x)  # 84 -> 10
        
        return x

---

## 3. Model Architecture

Define the Convolutional Neural Network (CNN) architecture. The network consists of:
- **2 Convolutional layers** with ReLU activation and MaxPooling
- **3 Fully connected layers** for classification

Input: RGB images (3 channels, 32x32 pixels)  
Output: 10 class probabilities (one per CIFAR-10 category)

In [ ]:
# Initialize the neural network
net = NeuralNetwork()

# Define loss function (CrossEntropyLoss for multi-class classification)
# Combines LogSoftmax and NLLLoss in one class
loss_function = nn.CrossEntropyLoss()

# Define optimizer (Stochastic Gradient Descent with momentum)
# lr=0.001: Learning rate (step size for weight updates)
# momentum=0.9: Helps accelerate gradients in the right direction
optimizer = optim.SGD(net.parameters(), lr=0.001, momentum=0.9)

In [ ]:
# Configure device (use GPU if available, otherwise CPU)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
net = net.to(device)
print(f'Training on {device}')

# Training loop: Run for 30 epochs (full passes through the training data)
for epoch in range(30):
    print(f'Training Epoch {epoch}...')
    
    # Track cumulative loss for this epoch
    running_loss = 0.0
    
    # Iterate over mini-batches of data
    # train_loader provides batches of 32 images and labels
    for i, data in enumerate(train_loader, 0):
        # Get batch of images and labels, move to device
        inputs, labels = data
        inputs, labels = inputs.to(device), labels.to(device)
        
        # Clear previous gradients
        optimizer.zero_grad()
        
        # Forward pass: Compute predicted outputs
        outputs = net(inputs)
        
        # Compute loss: Measure error between predictions and true labels
        loss = loss_function(outputs, labels)
        
        # Backward pass: Compute gradients of loss w.r.t. parameters
        loss.backward()
        
        # Update weights: Adjust parameters using computed gradients
        optimizer.step()
        
        # Accumulate loss for reporting
        running_loss += loss.item()
    
    # Print average loss for this epoch
    print(f'Epoch {epoch} Loss: {running_loss / len(train_loader):.4f}')

print('Training finished!')

---

## 4. Training

Train the neural network on the CIFAR-10 training set for 30 epochs. The training loop:
1. Forward pass: Compute predictions
2. Compute loss: Measure error
3. Backward pass: Compute gradients
4. Update weights: Adjust parameters via optimizer

In [ ]:
# Save the trained model parameters to disk
# This saves only the model weights, not the architecture
torch.save(net.state_dict(), 'trained_net.pth')

---

## 5. Model Persistence

Save the trained model to disk for later use, and optionally reload it.

In [ ]:
# Load a previously saved model
# Note: Must define the NeuralNetwork class first (see cell above)
net = NeuralNetwork()
net.load_state_dict(torch.load('trained_net.pth'))
net.eval()  # Set to evaluation mode (disables dropout, batch norm updates)

In [ ]:
# Evaluate model on test set
correct = 0
total = 0

# Set model to evaluation mode (disables training-specific operations)
net.eval()

# Disable gradient computation for inference (saves memory and computation)
with torch.no_grad():
    for data in test_loader:
        images, labels = data
        images, labels = images.to(device), labels.to(device)
        
        # Forward pass: Get predictions
        outputs = net(images)
        
        # Get predicted class (index with highest probability)
        _, predicted = torch.max(outputs, 1)
        
        # Update statistics
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

# Calculate and display accuracy
accuracy = 100 * correct / total
print(f'Accuracy of the network on the 10000 test images: {accuracy:.2f}%')

---

## 6. Model Evaluation

Evaluate the trained model on the test set to measure accuracy. This gives us an unbiased estimate of how well the model will perform on unseen data.

In [ ]:
# Image transformation pipeline for custom images
# Must match the training transformation for consistent results
new_transform = transforms.Compose([
    transforms.Resize((32, 32)),     # Resize to match CIFAR-10 dimensions
    transforms.ToTensor(),            # Convert to tensor
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))  # Normalize to [-1, 1]
])

def load_image(image_path):
    """
    Load and preprocess a single image for prediction.
    
    Args:
        image_path: Path to the image file
    
    Returns:
        Preprocessed tensor with batch dimension (1, 3, 32, 32)
    """
    image = Image.open(image_path)
    image = new_transform(image)      # Apply transformations
    image = image.unsqueeze(0)        # Add batch dimension
    return image

# List of image paths to classify
# Replace these with your actual image paths
image_path = ['example1.jpg', 'example2.jpg']
images = [load_image(img) for img in image_path]

# Make predictions
net.eval()  # Set to evaluation mode
with torch.no_grad():
    for image in images:
        output = net(image)
        _, predicted = torch.max(output, 1)
        print(f'Prediction: {class_names[predicted.item()]}')

---

## 7. Inference on Custom Images

Make predictions on your own images. The model expects RGB images that will be resized to 32x32 pixels.

**Note:** Current accuracy is ~68%, so some predictions will be incorrect. To improve accuracy, consider:
- Training for more epochs
- Using a deeper network
- Adding dropout for regularization
- Using data augmentation
- Transfer learning with pre-trained models